# GFBMD — fusion training and evaluation

Multimodal DeCAPS-Net (parsing maps + skeleton) on GFBMD with 5-fold subject-level cross-validation.

Prerequisites:
- GFBMD raw data: https://datadryad.org/dataset/doi:10.5061/dryad.s7h44j150
- Sapiens checkpoint `sapiens_1b_goliath_best_goliath_mIoU_7994_epoch_151_torchscript.pt2`: https://huggingface.co/facebook/sapiens-seg-1b-torchscript

Set the two paths below, then run the cells in order.

## Setup

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # run everything from the repository root
print(os.getcwd())

In [ ]:
import subprocess, sys

GFBMD_ROOT = '/path/to/GFBMD/Dataset'   # raw dataset root (Autism/ and Typical/ folders)
SAPIENS_CKPT = '/path/to/sapiens_1b_goliath_best_goliath_mIoU_7994_epoch_151_torchscript.pt2'

def run(*args):
    subprocess.run([sys.executable, *args], check=True)

## 1. Preprocessing
### 1.1 Parsing maps (YOLOv8 + Sapiens)
Downloads `yolov8n.pt` on first run. This is the slow step.

In [ ]:
run('preprocessing/gfbmd/01_extract_parsing.py',
    '--dataset-root', GFBMD_ROOT, '--output-root', 'data/gfbmd_parsing',
    '--sapiens-ckpt', SAPIENS_CKPT, '--device', 'cuda')

### 1.2 8.xlsx -> .skeleton files

In [ ]:
run('preprocessing/gfbmd/02_xlsx_to_skeleton.py',
    '--dataset-root', GFBMD_ROOT, '--out-root', 'data/gfbmd_ntu')

### 1.3 Denoise skeletons

In [ ]:
run('preprocessing/gfbmd/03_denoise_skeletons.py',
    '--skeleton-dir', 'data/gfbmd_ntu/nturgbd_raw/nturgb+d_skeletons120',
    '--names-txt', 'data/gfbmd_ntu/statistics/skes_available_name.txt',
    '--out-pkl', 'data/gfbmd_ntu/raw_denoised_joints.pkl')

### 1.4 Subject manifest
After this step check `data/gfbmd/manifest_summary.json`: all 100 subjects should have `fusion_ready=1`.

In [ ]:
run('preprocessing/gfbmd/04_build_manifest.py',
    '--parse-root', 'data/gfbmd_parsing', '--gfbmd-root', GFBMD_ROOT,
    '--skeleton-dir', 'data/gfbmd_ntu/nturgbd_raw/nturgb+d_skeletons120',
    '--statistics-dir', 'data/gfbmd_ntu/statistics',
    '--out-dir', 'data/gfbmd')

### 1.5 Skeleton cache

In [ ]:
run('preprocessing/gfbmd/05_build_cache.py',
    '--manifest-csv', 'data/gfbmd/subject_manifest.csv',
    '--skes-available-name-txt', 'data/gfbmd_ntu/statistics/skes_available_name.txt',
    '--raw-denoised-joints-pkl', 'data/gfbmd_ntu/raw_denoised_joints.pkl',
    '--out-h5', 'data/gfbmd/GFBMD_skeleton_cache.h5',
    '--out-summary-json', 'data/gfbmd/cache_summary.json')

### 1.6 Fixed 5-fold splits

In [ ]:
run('preprocessing/gfbmd/06_build_folds.py',
    '--manifest-csv', 'data/gfbmd/subject_manifest.csv',
    '--out-json', 'data/gfbmd/folds_5_seed42.json')

## 2. Train all 5 folds
Each fold saves `best.pt` (best validation epoch) under `work_dir/gfbmd_fusion/fold*/`.

In [ ]:
!python train_fusion.py --config configs/gfbmd_fusion.yaml

## 3. Evaluate from the best checkpoints

In [ ]:
!python train_fusion.py --config configs/gfbmd_fusion.yaml --phase test

In [ ]:
!python evaluate.py --dataset gfbmd --work-dir work_dir/gfbmd_fusion --out-dir work_dir/gfbmd_fusion/evaluation --num-folds 5

## 4. Metrics

In [ ]:
import json
with open('work_dir/gfbmd_fusion/evaluation/metrics.json') as f:
    metrics = json.load(f)
metrics